# Benchmarking wildfire spread models: the Hughes Fire

**SCIL, AI Collaborative for Wildfire, Workstream 2.** Red Sky demo, November 2026.

This notebook is the whole interface to the benchmarking framework. A **workflow** file goes in; scored, comparable model runs come out, and every step that produced them is recorded in Dagster on the WSTC cluster at NRP.

The case is the **Hughes Fire**: reported at 10:53 PST on January 22, 2025, on Lake Hughes Road near Castaic Lake in Los Angeles County, and 10,425 acres when contained. We run spread models on real LANDFIRE fuels and real station wind, then score each run against the observed final perimeter.

Read it top to bottom. Each section says what is about to happen and why, then does it.

| Section | What happens |
|---|---|
| 1. The workflow is a file | The YAML that defines the experiment |
| 2. From workflow to plan | The resolver checks it against the block registry and expands it into runs |
| 3. The graph | The plan drawn as the asset graph Dagster will run |
| 4. Identity and reuse | Why the same LANDFIRE is never downloaded twice |
| 5. Compile and push | The plan becomes one file and goes to the WSTC Dagster |
| 6. Inputs | LANDFIRE, wind, moisture, ignitions, the reference perimeter |
| 7. Runs | One model run per partition |
| 8. Results | Scores, previews, and where to look in Dagster |
| 9. Run it again | Nothing is recomputed |
| 10. What is real so far | An honest list |

## 0. Setup

The notebook runs anywhere that can reach the WSTC Dagster over HTTPS: a laptop, or NDP JupyterHub. It needs the repository (for the block registry and the resolver) and `pyyaml` and `requests`.

Two things are typed in, not stored: the WSTC dashboard password, and an email address. LANDFIRE's download service requires an email on every request; it travels as a run tag, so it is visible on the dashboard like any other run setting.

In [ ]:
import sys, getpass, time
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(REPO / "src"), str(REPO / "deploy" / "wstc")]

from bench.workflows import load_spec
from bench import wstc
from bench.notebook import (show_file, plan_tables, mermaid, identity_table, md_table,
                            run_partitions, asset_rows, show_previews)
from dagster_client import Dagster
from IPython.display import display, Markdown

ME = "apearce"                      # everything we put on the shared server starts with this
WORKFLOW = REPO / "workflows" / "hughes_red_sky.yaml"
BUILD = REPO / "deploy" / "wstc" / "build"     # generated files land here (not committed)
BUILD.mkdir(exist_ok=True)

In [ ]:
d = Dagster(password=getpass.getpass("WSTC Dagster password: "))
LFPS_EMAIL = input("Email for LANDFIRE requests: ").strip()
print(len(d.jobs()), "jobs on the server")

## 1. The workflow is a file

A workflow is configuration only, no code. It names a **case** (the fire, its working grid and time window), one or more **sources** for each input slot, the **models** to run, the **reference** to score against, and the **metrics**.

Every name in it refers to a **block**: a manifest in `blocks/` that declares what the block consumes, produces and accepts as parameters. Adding a model or a data source means adding a block, not changing the framework.

Things to notice in this one:

* `ignition` has **two sources**. That makes ignition a comparison axis: every model runs once per ignition.
* Giving a model parameter a **list** of values makes a sweep: one run per value (not used here; try `spread_adjustment: [1.2, 1.5]` on ELMFIRE).
* `wrf_fire` needs 3D atmospheric forcing that no input provides. We keep it in the file on purpose, to show what the resolver does with it.
* There is no WindNinja step: the wind comes from one weather station and is uniform over the domain.

In [ ]:
show_file(WORKFLOW)

## 2. From workflow to plan

The **resolver** turns the workflow into a plan, before anything runs. It

* checks every block exists and every parameter has the declared type (a typo fails here, not two hours into a run),
* merges parameters with the manifests' defaults,
* inserts automatic steps where a source's native format differs from the case (the ignition points arrive as WGS84 lat/lon, so each gets a reproject step to the case CRS, EPSG:5070),
* expands sweeps and comparison axes into individual runs,
* and skips combinations that cannot work, with the reason.

Then the WSTC compiler adds what the server needs: an asset key and an identity for every step (section 4).

In [ ]:
spec = load_spec(WORKFLOW)
reference_file = REPO / spec["reference"]["file"]
cp = wstc.compile_plan(spec, ME, reference_file.read_text() if reference_file.exists() else None)
plan_tables(cp)

## 3. The graph

This is the asset graph Dagster will hold: orange boxes are blocks that do real work (a grey dashed box would be a block that is still a dry run: it records what it would have done, nothing more). Each input step is an asset; `prediction` is one asset with one **partition per run**, which is what lets Dagster compare runs side by side; each metric is scored per partition; `report` gathers everything.

In [ ]:
mermaid(cp)

### Conventions at every block boundary

Models disagree about almost everything: which way is north in an array, whether wind direction is where it blows from or to, whether canopy cover is a percent or a fraction. The framework fixes one answer for each (`docs/conventions.md`) and makes every block convert in and out of it. Sources convert into the conventions, model adapters convert out of them, and nothing in between guesses.

| Rule | Value |
|---|---|
| Grid | the case CRS (EPSG:5070) at 30 m, **north-up**: row 0 is the northern edge |
| Wind direction | degrees clockwise from **grid** north, the direction the wind blows **from** |
| Units | SI: m, m/s, kg/m³, canopy cover as a fraction |
| Time | minutes since the case's t0 (the reported start); arrival time NaN = never burned |

Two examples from this run: the station-wind block rotates directions from true north to grid north (about 2.6° here, since EPSG:5070's central meridian is far east of Castaic), and the ELMFIRE adapter flips every grid on the way in and out, because pyretechnics puts row 0 at the **south** edge.

## 4. Identity and reuse

Every artifact gets an **identity**: a hash of everything that determines its content. That is the block's code, its version, its parameters and data selection, the parts of the case it depends on, and the identities of its inputs.

Artifacts are stored on the shared volume under their identity, not under the workflow's name. Before a step does any work it checks whether its identity already holds a finished artifact. If it does, it reuses it and says so.

So two workflows, or two runs, that need **the same LANDFIRE** (same version, layers and domain) share one download. LANDFIRE's identity deliberately ignores the parts of the case that cannot change it, like the ignition time. Change anything that matters, such as the domain, a parameter, or the block's code, and the identity changes, so the step runs again. Nothing stale is ever served.

In [ ]:
identity_table(cp)

## 5. Compile and push

The WSTC server takes **one Python file per upload** (at most 1 MB) and has only a handful of packages installed. We work within that with two files:

1. **`apearce_bench_env.py`** builds the blocks' Python environment once on the shared volume: Python 3.12, rasterio, NumPy 1.26 and a pre-compiled pyretechnics. Nearly all of its ~0.9 MB is pyretechnics. Running its job again when the environment exists does nothing.
2. **`apearce_hughes_red_sky.py`** is this workflow, compiled: the plan above and the blocks' code. Every step runs its block in that environment, in its own pod.

This is a stand-in. On the real deployment each block runs in its own container image, and only that line of the setup changes.

In [ ]:
env_file = wstc.build_env_file(ME, BUILD)
wf_file, cp = wstc.build_workflow_file(WORKFLOW, ME, BUILD)
jobs = wstc.job_names(ME, cp["workflow"])
for f in (env_file, wf_file):
    print(f"{f.name:32s} {f.stat().st_size:>9,} bytes (limit 1,000,000)")

In [ ]:
d.upload(str(env_file))
run_id, status, res = d.run(jobs["env"])
print(status, d.url_for(run_id))
md_table(asset_rows(res, ["env_id", "reused", "numpy_rasterio", "size_mb", "build_s"]))

In [ ]:
print([j for j in d.upload(str(wf_file)) if j.startswith(f"{ME}_{cp['workflow']}")])

## 6. Inputs

One job builds every input step and the reference perimeter. Independent steps run in parallel pods. The first time, this is where the waiting happens: LANDFIRE's service clips and reprojects the layers on its side, which can take a few minutes. Every step records what it did as metadata, which is what the table below reads back.

In [ ]:
t = time.time()
run_id, status, res = d.run(jobs["inputs"], tags={"bench/lfps_email": LFPS_EMAIL})
print(status, f"{time.time() - t:.0f} s", d.url_for(run_id))
if status != "SUCCESS":
    print("\n".join(d.errors(run_id)))
md_table(asset_rows(res, ["reused", "build_s", "grid", "station", "mean_wind_ms", "max_wind_ms",
                          "cells_changed", "time_utc", "reprojected", "burned_area_ha"]))

A closer look at two of them. **LANDFIRE** reports the products it requested and the most common fuel models on the case grid. **Station wind** reports which station it used and the wind over the window. Santa Ana conditions should show as strong wind from the northeast (around 30 to 60 degrees).

In [ ]:
for asset, md in res.items():
    if asset.endswith(("landfire/fetch", "station_wind/fetch")):
        display(Markdown(f"**{asset.split('/', 2)[-1]}**"))
        display(md_table([{"field": k, "value": v} for k, v in md.items()]))

## 7. Runs

Each run is one partition of `prediction`, plus its scores. We launch one Dagster run per partition, three at a time (the server allows four runs across all users).

Both are ELMFIRE (pyretechnics) on the full case grid, about 900 × 900 cells for 10 hours of fire, one per ignition.

In [ ]:
runs = run_partitions(d, jobs["runs"], sorted(cp["runs"]))
md_table([{"run": k, **v, "url": d.url_for(v["run_id"])} for k, v in runs.items()])

## 8. Results

The `report` asset gathers every finished run's scores into one table.

* **IoU** is the overlap between the predicted and the observed burned area (1 is perfect).
* **Area bias** is the predicted area relative to the observed one (0 is perfect, -0.5 means half the observed area).

In [ ]:
run_id, status, res = d.run(jobs["report"])
rep = next(md for a, md in res.items() if a.endswith("/report"))
md_table(rep["rows"])

And what the model runs look like. Every preview is drawn by the framework, not the model, with the same colours and fixed scales, so previews from different models and runs can be compared by eye: time of arrival in hours after t0, and fireline intensity on a log scale.

In [ ]:
previews = {}
for k, v in runs.items():
    for asset, md in d.results(v["run_id"]).items():
        if asset.endswith("/prediction") and "preview" in md:
            previews[k] = md["preview"]
show_previews(previews)

### Where to look in Dagster

The dashboard holds everything above, permanently, with provenance:

* **Assets > apearce > hughes_red_sky > prediction**: the partitions view compares runs; any numeric field (burned area, IoU, area bias) is charted across partitions.
* Every materialization records its identity, the artifact path on the shared volume, the run that built it, whether it was reused, and the full block summary.
* **Runs** has every launch with its logs, including each block's own output.

In [ ]:
print(f"{d.url}/assets/{ME}/{cp['workflow']}/prediction")

## 9. Run it again

Same workflow, same inputs: every input step should find its identity already built and reuse it in seconds, LANDFIRE included. This is the behaviour that makes a 500-fire benchmark affordable: shared inputs are fetched once, and adding a model means running only that model.

In [ ]:
t = time.time()
run_id, status, res = d.run(jobs["inputs"], tags={"bench/lfps_email": LFPS_EMAIL})
print(status, f"{time.time() - t:.0f} s for the whole input job")
md_table(asset_rows(res, ["reused", "built_by_run", "artifact"]))

## 10. What is real so far

| Part | Status |
|---|---|
| LANDFIRE 2024 fuels and canopy, LF 2020 elevation | real, from the LANDFIRE Product Service |
| Roads/urban override to GR2 | real |
| Station wind (nearest ASOS with data, hourly) | real, from the Iowa Environmental Mesonet (Sandberg, SDB, for Hughes); uniform over the domain (no WindNinja yet) |
| Fuel moisture | a fixed scenario value (1 h dead = 6%), other classes at the adapter's reported defaults |
| Ignition, reported origin | real (CAL FIRE incident page) |
| Ignition, NGFS first detection | **placeholder**: the same point until we have the detection |
| ELMFIRE (pyretechnics) | real |
| FARSITE | not in this workflow yet (its block is still a dry run) |
| WRF-Fire | skipped by the resolver (no 3D forcing) |
| Reference perimeter, IoU, area bias | real |

What changes for the real deployment: blocks run from their own container images instead of the shared environment, artifacts move to object storage, and many fires become a second partition dimension. The workflow files, the blocks' code, the conventions and the identity rules stay as they are.

### Cleanup (optional)

Removes our two files from the server; the jobs and assets disappear with them. Artifacts on the shared volume stay, so a later upload reuses them.

In [ ]:
# d.remove(env_file.name); d.remove(wf_file.name)